# Callable interfaces: lambdas, `*args`, and `**kwargs`

Design small callable adapters, unpack structured arguments safely, and build a transparent
model-evaluation workflow without hiding mistakes behind clever syntax.

**Lecture 2 · Python Foundations II · CMOR 438 / INDE 577**

## Python focus: model evaluation is the teaching context

This is a lesson about **function interfaces**. A small machine-learning-style evaluation harness
gives the syntax an authentic purpose:

| Python learning target | Evaluation task |
| --- | --- |
| lambda expression | define a tiny local predictor or sorting key |
| positional and keyword arguments | make call sites unambiguous |
| `*` at a call site | unpack a batch of residuals or evaluations |
| `*args` in a definition | accept a genuine variable-length collection |
| `**` at a call site | unpack a configuration mapping |
| `**kwargs` in a definition | collect genuinely open-ended run tags |
| argument forwarding | adapt or audit an existing callable |

The modeling examples are deliberately small. The goal is to decide when flexible call syntax makes
an interface clearer—and when it merely makes errors harder to detect.

## How to use this notebook

**Estimated time:** 45 minutes core, plus 35 minutes of practice and extension.

**Prerequisites:** Lecture 2 notebooks 00–01: functions as objects, type hints, NumPy-style
docstrings, exceptions, dataclasses, and pure functions.

Before running a call, identify which values bind to which parameters. Keep four questions separate:

1. Is the star syntax in a **function definition** or a **function call**?
2. Does it operate on positional arguments (`*`) or keyword arguments (`**`)?
3. What concrete object exists inside the function—a tuple or a dictionary?
4. Would an explicit parameter make the contract safer?

Run the notebook top to bottom in the **Rice DSM** kernel.

## Learning objectives

By the end of this notebook, you should be able to:

- distinguish parameters from arguments and all five Python parameter kinds;
- explain what a lambda expression creates and choose a named function when appropriate;
- diagnose late binding in lambdas created inside a loop;
- distinguish iterable unpacking at a call site from variadic parameter collection;
- predict the tuple stored in `args` and dictionary stored in `kwargs`;
- annotate, document, validate, and test variadic interfaces;
- explain why unrestricted `**kwargs` can conceal misspelled options;
- forward an existing callable's arguments while preserving its static signature; and
- apply these tools to an auditable model-evaluation workflow.

## Why this matters in industry

Data-science libraries are built around callables: transformations, loss functions, scoring rules,
callbacks, estimators, and configuration-driven jobs. Flexible argument handling makes these pieces
composable. It also creates common production failures:

- a misspelled hyperparameter disappears into `**kwargs`;
- a wrapper destroys the original signature and documentation;
- a lambda closes over a loop variable and every generated function behaves the same;
- an unpacked dictionary supplies the same parameter twice;
- a variadic interface accepts combinations the implementation cannot actually support.

The professional goal is not maximum flexibility. It is the **smallest honest interface** that
supports real callers and still lets editors, type checkers, tests, and humans detect mistakes.

## Professional practice: flexibility must preserve contracts

| Data scientist asks | Software engineer asks |
| --- | --- |
| What mathematical function is being passed? | What callable signature is promised? |
| Are residuals ordered and in compatible units? | Does `*args` validate every element? |
| Which settings produced this run? | Are stable settings explicit parameters? |
| Is the ranking criterion scientifically appropriate? | Is the key function deterministic and tested? |
| Could configuration change the conclusion? | Can duplicate or unknown keywords fail loudly? |
| Is this adapter changing the computation? | Does the wrapper preserve metadata and types? |

Call syntax is part of a public API—the supported interface other code is allowed to depend on.
Renaming a keyword-accessible parameter, accepting arbitrary
keywords, or changing positional order can break callers even when the function body is unchanged.

## 1. One call, five parameter kinds

Python supports positional-only, positional-or-keyword, variadic positional, keyword-only, and
variadic keyword parameters. The separators `/` and `*` make calling policy visible:

```text
def function(positional_only, /, ordinary, *args, keyword_only, **kwargs):
```

Use positional-only parameters when the name should not be part of that public calling contract, and
keyword-only parameters when
the name carries meaning or prevents swapped arguments. Do not add separators mechanically; use
them to communicate a stable contract.

In [ ]:
from collections.abc import Callable, Sequence
from dataclasses import dataclass
from functools import wraps
from inspect import signature
from math import isfinite, sqrt
from numbers import Real
from typing import ParamSpec, TypeVar

In [ ]:
def calibrate_reading(
    reading: object,
    /,
    *,
    scale: object,
    offset: object = 0.0,
) -> float:
    """Apply an affine calibration to one finite reading.

    Parameters
    ----------
    reading : object
        Raw finite measurement. This parameter is positional-only.
    scale : object
        Finite multiplicative calibration factor.
    offset : object, default=0.0
        Finite additive correction in output units.

    Returns
    -------
    float
        Calibrated value, computed as `scale * reading + offset`.

    Raises
    ------
    TypeError
        If an argument is not a real number or is a Boolean.
    ValueError
        If an argument is not finite.
    """

    values = {"reading": reading, "scale": scale, "offset": offset}
    finite_values: dict[str, float] = {}
    for name, value in values.items():
        if isinstance(value, bool) or not isinstance(value, Real):
            raise TypeError(f"{name} must be a real number; received {value!r}")
        finite_value = float(value)
        if not isfinite(finite_value):
            raise ValueError(f"{name} must be finite; received {value!r}")
        finite_values[name] = finite_value
    return (
        finite_values["scale"] * finite_values["reading"]
        + finite_values["offset"]
    )


calibrated_temperature = calibrate_reading(2.5, scale=1.8, offset=32.0)

assert calibrated_temperature == 36.5
print(signature(calibrate_reading))

The reading is positional-only because its role is obvious from proximity and its parameter name is
not intended as API. `scale` and `offset` are keyword-only because reversing them would silently
change the calculation. The signature makes the safe call style executable.

Python raises `TypeError` before entering the function if a caller violates this binding contract.
That is separate from our runtime validation of values after binding succeeds.

In [ ]:
for invalid_call in (
    lambda: calibrate_reading(reading=2.5, scale=1.8),
    lambda: calibrate_reading(2.5, 1.8, 32.0),
):
    try:
        invalid_call()
    except TypeError as error:
        print(type(error).__name__ + ":", error)
    else:
        raise AssertionError("the invalid calling convention should fail")

## 2. Lambda expressions create function objects

`lambda parameters: expression` evaluates to a function object. It is limited to one expression;
it cannot contain assignment statements, `try` blocks, or multiple `return` statements. It is not a
faster or more mathematical kind of function.

A lambda is useful when behavior is tiny, local, and clearer beside the operation that consumes it.
If behavior needs a meaningful name, documentation, validation, reuse, or direct tests, use `def`.

In [ ]:
def squared_residual(residual: float) -> float:
    """Return the square of one residual."""

    return residual**2


squared_residual_lambda = lambda residual: residual**2  # noqa: E731

assert squared_residual(-3.0) == 9.0
assert squared_residual_lambda(-3.0) == 9.0
assert callable(squared_residual_lambda)
print("named function:", squared_residual.__name__)
print("lambda:", squared_residual_lambda.__name__)

The behaviors match, but the interfaces do not communicate equally well. The named function has a
domain name and docstring. The lambda's generated name is `<lambda>`, which is less helpful in logs
and tracebacks. Assigning a lambda to a long-lived variable usually signals that `def` would be
clearer.

### A strong lambda use: a local sorting key

`sorted` needs a callable that maps each record to a comparison key. The transformation is short,
used once, and readable beside the sort. A tuple key provides deterministic tie-breaking.

In [ ]:
preliminary_scores = [
    {"model_name": "linear-small", "validation_loss": 0.18},
    {"model_name": "tree-depth-3", "validation_loss": 0.14},
    {"model_name": "linear-wide", "validation_loss": 0.18},
]

ranked_scores = sorted(
    preliminary_scores,
    key=lambda result: (result["validation_loss"], result["model_name"]),
)

assert ranked_scores[0]["model_name"] == "tree-depth-3"
assert [result["model_name"] for result in ranked_scores[1:]] == [
    "linear-small",
    "linear-wide",
]

### The late-binding trap

Functions created in a loop close over the **name**, not a frozen copy of its current value. The
loop finishes with `coefficient == 3`, so every lambda below reads 3 when called.

**Predict the two result lists before running.** The corrected version uses a default parameter to
bind the current object at function-creation time. This is a targeted technique, not a reason to use
defaults casually.

In [ ]:
late_bound_scalers = [
    lambda value: coefficient * value  # noqa: B023 - deliberate late binding
    for coefficient in (1.0, 2.0, 3.0)
]
bound_scalers = [
    lambda value, coefficient=coefficient: coefficient * value
    for coefficient in (1.0, 2.0, 3.0)
]

late_results = [transform(10.0) for transform in late_bound_scalers]
bound_results = [transform(10.0) for transform in bound_scalers]

assert late_results == [30.0, 30.0, 30.0]
assert bound_results == [10.0, 20.0, 30.0]

If the generated functions are important, prefer a named factory with a documented closure. The
late-binding behavior is a Python closure rule, not a special defect in lambda expressions; nested
functions created with `def` capture names the same way.

## 3. `*` means unpack at a call and collect in a definition

The same symbol has inverse roles:

- `function(*values)` **unpacks** an iterable into separate positional arguments at the call site.
- `def function(*values)` **collects** extra positional arguments into a tuple inside the function.

The conventional name `args` has no magic. A domain name such as `residuals` is often clearer.
Variadic parameters are appropriate only when zero or more values play the same role.

In [ ]:
def root_mean_square(*residuals: object) -> float:
    """Compute root mean square for one or more finite residuals.

    Parameters
    ----------
    *residuals : object
        Residuals in compatible units. Each must be a finite real number.

    Returns
    -------
    float
        Nonnegative root mean square in the residuals' units.

    Raises
    ------
    ValueError
        If no residual is provided or a residual is not finite.
    TypeError
        If a residual is not a real number or is a Boolean.
    """

    if not residuals:
        raise ValueError("root_mean_square requires at least one residual")
    finite_residuals: list[float] = []
    for index, residual in enumerate(residuals):
        if isinstance(residual, bool) or not isinstance(residual, Real):
            raise TypeError(
                f"residuals[{index}] must be a real number; received {residual!r}"
            )
        finite_residual = float(residual)
        if not isfinite(finite_residual):
            raise ValueError(
                f"residuals[{index}] must be finite; received {residual!r}"
            )
        finite_residuals.append(finite_residual)
    mean_square = sum(value**2 for value in finite_residuals) / len(residuals)
    return sqrt(mean_square)


residual_batch = (-1.0, 2.0, -2.0)
batch_rms = root_mean_square(*residual_batch)

assert batch_rms == sqrt(3.0)
assert isinstance(residual_batch, tuple)

Inside `root_mean_square`, `residuals` is a tuple even if the caller unpacks a list, tuple, or
generator. The annotation `*residuals: object` describes the type of **each collected argument**, not
the tuple itself. Runtime validation narrows every element to a finite float.

Without the star, `root_mean_square(residual_batch)` passes one tuple as one argument. That violates
the element contract rather than unpacking it.

In [ ]:
try:
    root_mean_square(residual_batch)
except TypeError as error:
    print(type(error).__name__ + ":", error)

try:
    root_mean_square()
except ValueError as error:
    print(type(error).__name__ + ":", error)

### Variadic callables can express ensembles

An ensemble genuinely accepts a variable number of predictors that share one interface. The
positional-only observation is followed by `*predictors`; parameters after a variadic positional
parameter would automatically be keyword-only.

In [ ]:
def mean_ensemble_prediction(
    input_value: object,
    /,
    *predictors: Callable[[float], float],
) -> float:
    """Average predictions from one or more scalar predictors.

    Parameters
    ----------
    input_value : object
        Finite scalar input passed to every predictor.
    *predictors : callable
        One or more callables accepting and returning a float.

    Returns
    -------
    float
        Arithmetic mean of finite predictions.

    Raises
    ------
    ValueError
        If no predictor is supplied or a numeric value is not finite.
    TypeError
        If an input, predictor, or prediction has an incompatible type.
    """

    calibrated_input = calibrate_reading(input_value, scale=1.0)
    if not predictors:
        raise ValueError("mean_ensemble_prediction requires at least one predictor")
    predictions: list[float] = []
    for index, predictor in enumerate(predictors):
        if not callable(predictor):
            raise TypeError(f"predictors[{index}] must be callable")
        prediction = predictor(calibrated_input)
        predictions.append(calibrate_reading(prediction, scale=1.0))
    return sum(predictions) / len(predictions)


ensemble_value = mean_ensemble_prediction(
    2.0,
    lambda value: 2.0 * value + 1.0,
    lambda value: 1.5 * value + 1.5,
)

assert ensemble_value == 4.75

## 4. `**` means unpack at a call and collect in a definition

Again the call-site and definition roles are inverse:

- `function(**configuration)` unpacks a mapping as named arguments.
- `def function(**tags)` collects unmatched keyword arguments into a new dictionary.

Keyword keys must be strings. Parameters explicitly named in the signature bind first; only
remaining keywords enter `tags`. Use `**kwargs` when the vocabulary is genuinely open or when an
adapter must forward another signature—not merely to avoid designing an interface.

In [ ]:
def record_run(
    run_id: str,
    /,
    *,
    dataset_version: str,
    seed: int,
    **tags: object,
) -> dict[str, object]:
    """Create metadata for a reproducible experiment run.

    Parameters
    ----------
    run_id : str
        Nonblank run identifier.
    dataset_version : str
        Nonblank immutable dataset version or content identifier.
    seed : int
        Reproducibility seed; Booleans are rejected.
    **tags : object
        Open-ended descriptive metadata. Tag names must be nonblank.

    Returns
    -------
    dict of str to object
        New metadata dictionary; caller-owned mappings are not mutated.

    Raises
    ------
    TypeError
        If required text or integer fields have incompatible types.
    ValueError
        If required text or a tag name is blank.
    """

    if not isinstance(run_id, str):
        raise TypeError(f"run_id must be a string; received {run_id!r}")
    if not isinstance(dataset_version, str):
        raise TypeError("dataset_version must be a string")
    normalized_run_id = run_id.strip()
    normalized_version = dataset_version.strip()
    if not normalized_run_id:
        raise ValueError("run_id must not be blank")
    if not normalized_version:
        raise ValueError("dataset_version must not be blank")
    if isinstance(seed, bool) or not isinstance(seed, int):
        raise TypeError(f"seed must be an integer; received {seed!r}")
    if any(not name.strip() for name in tags):
        raise ValueError("tag names must not be blank")
    return {
        "run_id": normalized_run_id,
        "dataset_version": normalized_version,
        "seed": seed,
        **tags,
    }

In [ ]:
required_configuration = {"dataset_version": "sha256:8f3a", "seed": 438}
descriptive_tags = {"owner": "model-risk", "purpose": "lecture-demo"}

run_metadata = record_run(
    "run-017",
    **required_configuration,
    **descriptive_tags,
)

assert run_metadata["seed"] == 438
assert run_metadata["owner"] == "model-risk"
assert "owner" not in required_configuration

Stable, behavior-changing settings such as `dataset_version` and `seed` are explicit keyword-only
parameters. Truly open-ended descriptive tags use `**tags`. This distinction lets Python reject a
misspelled required setting while preserving extensible metadata.

Even open tags need a governance policy in a real system: serializable value types, privacy rules,
allowed cardinality, and naming conventions.

### Duplicate values fail before the function body

If two unpacked mappings provide the same keyword—or an explicit keyword duplicates one from a
mapping—Python raises `TypeError`. Build one configuration first when override behavior is intended.
Dictionary union `defaults | overrides` makes the right-hand precedence explicit.

In [ ]:
defaults = {"dataset_version": "development", "seed": 0}
overrides = {"seed": 577}
resolved_configuration = defaults | overrides

assert resolved_configuration == {
    "dataset_version": "development",
    "seed": 577,
}
assert record_run("run-018", **resolved_configuration)["seed"] == 577

try:
    record_run("run-019", seed=1, **{"dataset_version": "v2", "seed": 2})
except TypeError as error:
    print(type(error).__name__ + ":", error)

### Why unrestricted `**kwargs` is dangerous

If `record_run` accepted every option only through `**kwargs`, `datset_version="v2"` could become an
innocent-looking unused key. Explicit parameters give editors and static checkers more information
and let Python reject unknown names.

For a closed but heterogeneous keyword schema, Python also offers `TypedDict` with `Unpack`. That can
describe known `**kwargs` to a type checker, but it still does not perform runtime validation. We
reserve that advanced pattern for APIs that must expose keyword unpacking rather than ordinary
explicit parameters.

## 5. Forwarding another callable's interface

Wrappers often receive `*args` and `**kwargs` only to pass them onward. Annotating them as `object`
loses the relationship to the wrapped callable. `ParamSpec` captures its positional and keyword
parameter specification, while a type variable captures its return type.

In [ ]:
P = ParamSpec("P")
R = TypeVar("R")


def audited_call(  # noqa: UP047 - explicit ParamSpec is the lesson
    function: Callable[P, R],
    /,
    *args: P.args,
    **kwargs: P.kwargs,
) -> R:
    """Call a function while reporting its name and argument counts.

    Parameters
    ----------
    function : callable
        Callable whose signature and return type are preserved by `ParamSpec` and `R`.
    *args : object
        Positional arguments accepted by `function`.
    **kwargs : object
        Keyword arguments accepted by `function`.

    Returns
    -------
    object
        The original result, with its static type preserved.

    Notes
    -----
    This teaching audit prints only counts, never values. Production logs must
    not expose secrets, personal data, proprietary samples, or credentials.
    """

    function_name = getattr(function, "__name__", type(function).__name__)
    print(
        f"calling {function_name}: "
        f"{len(args)} positional, {len(kwargs)} keyword"
    )
    return function(*args, **kwargs)


audited_temperature = audited_call(
    calibrate_reading,
    2.5,
    scale=1.8,
    offset=32.0,
)

assert audited_temperature == 36.5

`P.args` and `P.kwargs` are for annotations on forwarding parameters. At runtime, `args` is still a
tuple and `kwargs` is still a dictionary. Static tools can connect the wrapper call to the original
signature; runtime binding and validation still happen normally.

## 6. Worked example: evaluate and rank candidate models

We now combine the ideas in a small regression evaluation. Each candidate holds a scalar predictor.
An evaluator computes residuals, accepts an explicit dataset version and seed, permits descriptive
tags, and returns an immutable result. A ranking function accepts a variable number of results and a
local key callable.

This is not a training system. The held-out observations are fixed teaching data, and selecting a
model repeatedly on the same validation set would eventually overfit that set.

In [ ]:
@dataclass(frozen=True, slots=True)
class Observation:
    """Pair one finite scalar input with one finite regression target.

    Parameters
    ----------
    input_value : float
        Finite scalar model input.
    target : float
        Finite observed regression target.

    Raises
    ------
    TypeError
        If a value is not a real number or is a Boolean.
    ValueError
        If a value is not finite.
    """

    input_value: float
    target: float

    def __post_init__(self) -> None:
        object.__setattr__(
            self,
            "input_value",
            calibrate_reading(self.input_value, scale=1.0),
        )
        object.__setattr__(
            self,
            "target",
            calibrate_reading(self.target, scale=1.0),
        )


@dataclass(frozen=True, slots=True)
class CandidateModel:
    """Name one scalar prediction function.

    Parameters
    ----------
    name : str
        Nonblank model identifier.
    predictor : callable
        Function accepting and returning one scalar float.

    Raises
    ------
    TypeError
        If `name` is not text or `predictor` is not callable.
    ValueError
        If `name` is blank.
    """

    name: str
    predictor: Callable[[float], float]

    def __post_init__(self) -> None:
        if not isinstance(self.name, str):
            raise TypeError(f"name must be a string; received {self.name!r}")
        if not self.name.strip():
            raise ValueError("name must not be blank")
        if not callable(self.predictor):
            raise TypeError("predictor must be callable")
        object.__setattr__(self, "name", self.name.strip())


@dataclass(frozen=True, slots=True)
class ModelEvaluation:
    """Store one model's validated evaluation result and provenance.

    Parameters
    ----------
    model_name : str
        Nonblank candidate-model identifier.
    root_mean_square_error : float
        Finite, nonnegative error in target units.
    metadata : tuple of tuple
        Immutable `(name, value)` provenance pairs with string names.

    Raises
    ------
    TypeError
        If a field has an incompatible runtime type.
    ValueError
        If the name is blank or the error is negative or nonfinite.
    """

    model_name: str
    root_mean_square_error: float
    metadata: tuple[tuple[str, object], ...]

    def __post_init__(self) -> None:
        if not isinstance(self.model_name, str):
            raise TypeError("model_name must be a string")
        normalized_name = self.model_name.strip()
        if not normalized_name:
            raise ValueError("model_name must not be blank")
        finite_error = calibrate_reading(
            self.root_mean_square_error,
            scale=1.0,
        )
        if finite_error < 0.0:
            raise ValueError(
                "root_mean_square_error must be nonnegative; "
                f"received {self.root_mean_square_error!r}"
            )
        if not isinstance(self.metadata, tuple):
            raise TypeError("metadata must be a tuple of (name, value) pairs")
        for index, item in enumerate(self.metadata):
            if (
                not isinstance(item, tuple)
                or len(item) != 2
                or not isinstance(item[0], str)
            ):
                raise TypeError(
                    f"metadata[{index}] must be a (string, value) tuple"
                )
        object.__setattr__(self, "model_name", normalized_name)
        object.__setattr__(self, "root_mean_square_error", finite_error)

In [ ]:
def evaluate_model(
    candidate: CandidateModel,
    observations: Sequence[Observation],
    /,
    *,
    dataset_version: str,
    seed: int,
    metric: Callable[..., float] = root_mean_square,
    **tags: object,
) -> ModelEvaluation:
    """Evaluate one scalar model on a fixed observation sequence.

    Parameters
    ----------
    candidate : CandidateModel
        Named scalar predictor.
    observations : sequence of Observation
        Nonempty evaluation data in deterministic order.
    dataset_version : str
        Immutable identifier for the evaluation data.
    seed : int
        Recorded reproducibility seed; evaluation here is deterministic.
    metric : callable, default=root_mean_square
        Variadic residual metric returning a finite scalar.
    **tags : object
        Descriptive run metadata.

    Returns
    -------
    ModelEvaluation
        Model name, metric result, and sorted provenance items.

    Raises
    ------
    ValueError
        If observations are empty or a numeric result is not finite.
    TypeError
        If an observation or callable has an incompatible runtime type.
    """

    if not observations:
        raise ValueError("observations must contain at least one item")
    if not callable(metric):
        raise TypeError("metric must be callable")
    residuals: list[float] = []
    for index, observation in enumerate(observations):
        if not isinstance(observation, Observation):
            raise TypeError(f"observations[{index}] must be an Observation")
        prediction = calibrate_reading(
            candidate.predictor(observation.input_value),
            scale=1.0,
        )
        residuals.append(prediction - observation.target)
    metric_value = calibrate_reading(metric(*residuals), scale=1.0)
    metadata = record_run(
        f"evaluation-{candidate.name}",
        dataset_version=dataset_version,
        seed=seed,
        **tags,
    )
    return ModelEvaluation(
        model_name=candidate.name,
        root_mean_square_error=metric_value,
        metadata=tuple(sorted(metadata.items())),
    )


def rank_evaluations(
    *evaluations: ModelEvaluation,
    key: Callable[[ModelEvaluation], object],
) -> list[ModelEvaluation]:
    """Return model evaluations ordered by a caller-provided criterion.

    Parameters
    ----------
    *evaluations : ModelEvaluation
        One or more evaluation results.
    key : callable
        Keyword-only function mapping an evaluation to a sortable key.

    Returns
    -------
    list of ModelEvaluation
        New list ordered in ascending key order.

    Raises
    ------
    ValueError
        If no evaluation is supplied.
    TypeError
        If an item is not a `ModelEvaluation` or `key` is not callable.
    """

    if not evaluations:
        raise ValueError("rank_evaluations requires at least one evaluation")
    if not callable(key):
        raise TypeError("key must be callable")
    if not all(isinstance(item, ModelEvaluation) for item in evaluations):
        raise TypeError("every evaluation must be a ModelEvaluation")
    return sorted(evaluations, key=key)

In [ ]:
validation_observations = (
    Observation(0.0, 1.1),
    Observation(1.0, 2.9),
    Observation(2.0, 5.2),
    Observation(3.0, 6.8),
)
candidate_models = (
    CandidateModel("linear-theory", lambda value: 2.0 * value + 1.0),
    CandidateModel("linear-shrunk", lambda value: 1.7 * value + 1.2),
    CandidateModel("constant-baseline", lambda _value: 4.0),
)
evaluation_config = {
    "dataset_version": "validation-v1",
    "seed": 438,
    "owner": "course-demo",
}

evaluations = tuple(
    evaluate_model(
        candidate,
        validation_observations,
        **evaluation_config,
    )
    for candidate in candidate_models
)
ranking = rank_evaluations(
    *evaluations,
    key=lambda result: (
        result.root_mean_square_error,
        result.model_name,
    ),
)

for result in ranking:
    print(f"{result.model_name:<18} RMSE={result.root_mean_square_error:.4f}")

assert ranking[0].model_name == "linear-theory"
assert ranking[-1].model_name == "constant-baseline"
assert dict(ranking[0].metadata)["dataset_version"] == "validation-v1"

### Interpret the workflow

- Lambdas are confined to tiny predictors and one local ranking key.
- `*residuals` matches a metric whose arguments genuinely share one role.
- `metric(*residuals)` is call-site unpacking, not variadic collection.
- Stable provenance is explicit; open descriptive tags are collected separately.
- `rank_evaluations(*evaluations, key=...)` combines positional unpacking with a required
  keyword-only criterion.
- Every flexible boundary still validates what it can promise.

RMSE alone does not establish generalization, fairness, calibration, causal validity, or deployment
safety. The interface makes one computation reproducible; it does not answer every modeling question.

## Debugging argument binding

Use this sequence when a flexible call fails:

1. Write the target signature and label `/`, `*args`, keyword-only fields, and `**kwargs`.
2. Inspect `signature(function)` if the callable preserves one.
3. Expand call-site data on paper: what arguments does `*sequence` emit, and what names does
   `**mapping` emit?
4. Check for duplicate parameter values before debugging the function body.
5. Print `repr(args)` and `repr(kwargs)` in a minimal reproduction—never in logs containing secrets.
6. Separate binding `TypeError` from runtime validation errors raised inside the function.
7. For loop-created functions, inspect which external names they close over.
8. Replace a complicated lambda with `def`, a name, a docstring, and a direct unit test.

## Common failure modes

| Failure | Cause | Better design |
| --- | --- | --- |
| every loop lambda uses the final value | closures resolve the shared name later | bind deliberately or use a factory |
| traceback contains only `<lambda>` | important behavior has no domain name | use `def` |
| tuple passed where numbers were expected | call omitted `*` unpacking | distinguish one argument from many |
| `TypeError: multiple values` | keyword appeared twice | resolve configuration before the call |
| misspelled option is silently accepted | unrestricted `**kwargs` swallowed it | use explicit keyword-only parameters |
| wrapper accepts anything statically | signature relationship was erased | use `ParamSpec` when forwarding |
| metric accepts an empty collection | variadic lower bound was unstated | validate cardinality |
| logs leak data or secrets | wrapper records raw arguments | log only approved metadata |
| ranking is reproducible but misleading | scientific criterion is inadequate | test assumptions and report limitations |

## Guided practice: build a polynomial factory

Implement `make_polynomial(*coefficients)` using coefficients in increasing power order:

$$
p(x) = c_0 + c_1x + c_2x^2 + \cdots.
$$

The factory should validate at least one finite coefficient, freeze the coefficients in a tuple,
and return a named nested function—not a lambda—evaluated with Horner's method. The returned function
must validate a finite input.

**Success criteria:** constant, linear, and quadratic examples work; an empty coefficient list,
nonfinite coefficient, and invalid input fail with specific messages; later mutation of a caller's
original list cannot change the polynomial.

In [ ]:
def make_polynomial(*coefficients: object) -> Callable[[object], float]:
    """Construct a polynomial from coefficients in increasing power order.

    Parameters
    ----------
    *coefficients : object
        One or more finite real coefficients, starting with the constant term.

    Returns
    -------
    callable
        Function accepting one finite real input and returning the polynomial value.

    Raises
    ------
    ValueError
        If no coefficient is supplied or a coefficient is not finite.
    TypeError
        If a coefficient is not a real number or is a Boolean.
    """

    if not coefficients:
        raise ValueError("make_polynomial requires at least one coefficient")
    frozen_coefficients = tuple(
        calibrate_reading(coefficient, scale=1.0)
        for coefficient in coefficients
    )

    def polynomial(input_value: object) -> float:
        """Evaluate the constructed polynomial at one finite input."""

        finite_input = calibrate_reading(input_value, scale=1.0)
        result = 0.0
        for coefficient in reversed(frozen_coefficients):
            result = result * finite_input + coefficient
        return result

    return polynomial


quadratic = make_polynomial(1.0, -3.0, 2.0)

assert quadratic(0.0) == 1.0
assert quadratic(1.0) == 0.0
assert quadratic(2.0) == 3.0

### Guided reflection

- Why is `*coefficients` honest here, while `**kwargs` for coefficient names would be awkward?
- What tuple exists inside the factory?
- Why does freezing the coefficients matter for reproducibility?
- Why is the returned callable a named nested function rather than a lambda?

**Checkpoint:** explain definition-site collection, closure capture, and call-site invocation as three
separate events.

## Independent practice: weighted ensemble adapter

Design `make_weighted_ensemble(*predictors, weights)` where `weights` is required keyword-only.
Validate that there is at least one callable, weights are finite and nonnegative, lengths match, and
positive total weight exists. Return a named callable that computes the normalized weighted mean.

**Success criteria:** test two predictors by hand; test one predictor; reject no predictors, length
mismatch, negative weight, all-zero weights, a non-callable predictor, and a nonfinite prediction.
Document the returned function's input and output units. Explain why `weights` is keyword-only.

In [ ]:
# Starter checks for your implementation. Define the function, then remove the guard.
student_ensemble_factory = globals().get("make_weighted_ensemble")

if callable(student_ensemble_factory):
    weighted_model = student_ensemble_factory(
        lambda value: value,
        lambda value: 2.0 * value,
        weights=(1.0, 3.0),
    )
    assert weighted_model(4.0) == 7.0
else:
    print("Define make_weighted_ensemble to activate the independent checks.")

## Extension: a signature-preserving decorator

A decorator is a callable that receives a function and returns a replacement. Implement
`require_finite_result` using `ParamSpec`, `TypeVar`, `*args`, and `**kwargs`. Its wrapper should call
the original function, reject a non-real or nonfinite result, and preserve metadata with
`functools.wraps`.

**Success criteria:** the decorated function keeps its `__name__`, `__doc__`, and inspectable
signature; valid calls are unchanged; invalid results fail. State why a wrapper must avoid logging raw
arguments by default.

In [ ]:
def require_finite_result(  # noqa: UP047 - explicit ParamSpec is the lesson
    function: Callable[P, float],
) -> Callable[P, float]:
    """Decorate a callable to require a finite real result."""

    @wraps(function)
    def checked(*args: P.args, **kwargs: P.kwargs) -> float:
        result = function(*args, **kwargs)
        return calibrate_reading(result, scale=1.0)

    return checked


@require_finite_result
def reciprocal(value: float, *, scale: float = 1.0) -> float:
    """Return `scale / value`."""

    return scale / value


assert reciprocal(2.0, scale=4.0) == 2.0
assert reciprocal.__name__ == "reciprocal"
assert "scale" in str(signature(reciprocal))

## Extension: `functools.partial` versus lambda

`functools.partial` creates a callable with selected arguments pre-filled. Compare these adapters:

```python
to_fahrenheit_lambda = lambda reading: calibrate_reading(
    reading, scale=1.8, offset=32.0
)

from functools import partial
to_fahrenheit_partial = partial(
    calibrate_reading, scale=1.8, offset=32.0
)
```

Inspect their signatures, names, representations, and tracebacks. Neither automatically supplies a
domain-quality name and docstring. For a public conversion function, a small documented `def` may
still be the clearest interface.

## Retrieval practice

Answer without executing code:

1. What are Python's five parameter kinds?
2. How do `/` and `*` change the calling contract?
3. What object does a lambda expression create, and what syntax can its body contain?
4. Why do lambdas created in a loop often share the final loop value?
5. Contrast `metric(*residuals)` with `def metric(*residuals)`.
6. What runtime types do `args` and `kwargs` have inside a function?
7. Why can unrestricted `**kwargs` conceal an API typo?
8. When is an open-ended keyword vocabulary legitimate?
9. What does `ParamSpec` preserve that `Callable[..., R]` does not?
10. Why does reproducible ranking not guarantee a scientifically valid model choice?

## Takeaway

Lambdas, `*args`, and `**kwargs` are tools for adapting callables—not measures of Python fluency. Use
a lambda for tiny local behavior, variadic positional arguments for values with one repeated role,
and variadic keywords only for a genuinely open vocabulary or faithful forwarding.

Keep stable options explicit, make meaningful settings keyword-only, validate every collected value,
and preserve wrapped signatures with `ParamSpec` and `wraps`. The strongest interface is usually the
least flexible one that accurately serves its callers.

## Connection to the next notebook

The next notebook loads native text, CSV, JSON, and JSON Lines. Configuration mappings will become
real file records, and flexible dictionaries will cross a trust boundary. We will preserve raw data,
parse explicit schemas, and construct validated objects instead of forwarding arbitrary fields into
constructors with `**record`.

## Further reading

- [Python tutorial: special parameters, arbitrary arguments, unpacking, and lambdas](https://docs.python.org/3/tutorial/controlflow.html#more-on-defining-functions)
- [Python expression reference: calls](https://docs.python.org/3/reference/expressions.html#calls)
- [Python typing specification: callable signatures](https://typing.python.org/en/latest/spec/callables.html)
- [Python `functools`: higher-order callable tools](https://docs.python.org/3/library/functools.html)
- [PEP 612: parameter specification variables](https://peps.python.org/pep-0612/)
- [PEP 692: typed `**kwargs`](https://peps.python.org/pep-0692/)

Treat these as references. Interface quality still depends on domain meaning, explicit policy, and
tests at real boundaries.